# Backtesting de Modelos de Risco - Solution

O **backtesting** e o processo de avaliar a qualidade de um modelo de VaR comparando
suas previsoes com os retornos realizados. Um bom modelo deve ter uma taxa de violacao
(retornos abaixo do VaR) proxima ao nivel de confianca $\alpha$.

Alem da frequencia de violacoes, verificamos se elas sao **independentes** no tempo
(sem clusters) e se a distribuicao prevista e correta.

**Conteudo:**
1. Teste de Kupiec (1995) — cobertura incondicional
2. Teste de Christoffersen (1998) — cobertura condicional (independencia)
3. Teste de Berkowitz (2001) — baseado em PIT
4. Traffic light system (Basel)
5. Backtesting comparativo
6. Dynamic Quantile (DQ) test — Engle e Manganelli (2004)

**Referencias:**
- Kupiec, P. (1995). Techniques for Verifying the Accuracy of Risk Measurement Models. *Journal of Derivatives*.
- Christoffersen, P. (1998). Evaluating Interval Forecasts. *International Economic Review*.
- Berkowitz, J. (2001). Testing Density Forecasts, with Applications to Risk Management. *Journal of Business & Economic Statistics*.
- Engle, R. & Manganelli, S. (2004). CAViaR: Conditional Autoregressive Value at Risk by Regression Quantiles. *Journal of Business & Economic Statistics*.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from archbox.models import GARCH
from archbox.risk import EWMA

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)

# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']
n = len(returns)

# Estimar modelos para gerar VaR series
# 1. GARCH-Normal
model_n = GARCH(returns.values, p=1, q=1, mean='constant', dist='normal')
res_n = model_n.fit(disp=False)
mu_n = res_n.params[0]

# 2. GARCH-t
model_t = GARCH(returns.values, p=1, q=1, mean='constant', dist='studentt')
res_t = model_t.fit(disp=False)
mu_t = res_t.params[0]
nu = res_t.params[-1]

# 3. EWMA
ewma = EWMA(returns.values, lam=0.94)
res_ewma = ewma.fit()

# Gerar series de VaR (99%)
alpha = 0.01
z_alpha = stats.norm.ppf(alpha)
t_alpha = stats.t.ppf(alpha, df=nu) * np.sqrt((nu - 2) / nu)

var_normal = mu_n + z_alpha * res_n.conditional_volatility
var_garch_t = mu_t + t_alpha * res_t.conditional_volatility
var_ewma = returns.mean() + z_alpha * res_ewma.conditional_volatility

print(f"Modelos estimados. VaR series geradas para alpha = {alpha}")
print(f"GARCH-t: nu = {nu:.2f}")

## 1. Teste de Kupiec (1995)

O teste de **cobertura incondicional** (Proportion of Failures - POF) verifica se
a taxa de violacao observada $\hat{p}$ e consistente com o nivel nominal $\alpha$.

**Hipoteses:**
- $H_0: p = \alpha$ (o modelo esta corretamente calibrado)
- $H_1: p \neq \alpha$

Estatistica de teste (razao de verossimilhanca):

$$LR_{uc} = -2 \ln\left[\frac{\alpha^x (1-\alpha)^{T-x}}{\hat{p}^x (1-\hat{p})^{T-x}}\right] \sim \chi^2(1)$$

onde $x$ e o numero de violacoes e $T$ e o numero total de observacoes.

In [ ]:
def kupiec_test(returns_arr, var_arr, alpha):
    """Teste de Kupiec (1995) - Cobertura Incondicional (POF)."""
    violations = returns_arr < var_arr
    x = violations.sum()  # numero de violacoes
    T = len(returns_arr)
    p_hat = x / T  # taxa observada

    # Evitar log(0)
    if x == 0 or x == T:
        return {'statistic': np.inf, 'pvalue': 0.0, 'violations': x,
                'rate': p_hat, 'expected_rate': alpha}

    # LR statistic
    lr = -2 * (x * np.log(alpha) + (T - x) * np.log(1 - alpha)
               - x * np.log(p_hat) - (T - x) * np.log(1 - p_hat))

    pvalue = 1 - stats.chi2.cdf(lr, df=1)

    return {'statistic': lr, 'pvalue': pvalue, 'violations': x,
            'rate': p_hat, 'expected_rate': alpha}

# Aplicar teste de Kupiec para cada modelo
print("=== Teste de Kupiec (Cobertura Incondicional) - VaR 99% ===\n")
print(f"{'Modelo':<20} {'Violacoes':>10} {'Taxa':>8} {'Esperada':>10} {'LR':>10} {'p-valor':>10} {'Decisao':>12}")
print("=" * 82)

for name, var_series in [('GARCH-Normal', var_normal),
                          ('GARCH-t', var_garch_t),
                          ('EWMA', var_ewma)]:
    result = kupiec_test(returns.values, var_series, alpha)
    decision = "Rejeita H0" if result['pvalue'] < 0.05 else "Nao rejeita"
    print(f"{name:<20} {result['violations']:>10} {result['rate']:>8.4f} "
          f"{result['expected_rate']:>10.4f} {result['statistic']:>10.4f} "
          f"{result['pvalue']:>10.4f} {decision:>12}")

print("\nSe p-valor < 0.05: rejeita H0 (modelo mal calibrado)")
print("Se p-valor >= 0.05: nao rejeita H0 (taxa de violacao consistente com alpha)")

## 2. Teste de Christoffersen (1998)

O teste de Kupiec verifica apenas a **frequencia** das violacoes. O teste de
**Christoffersen** adiciona um teste de **independencia**: as violacoes devem ser
i.i.d., sem clusters.

Seja $I_t = \mathbb{1}(r_t < \text{VaR}_t)$ o indicador de violacao. O teste verifica
se a probabilidade de violacao em $t$ depende da violacao em $t-1$:

$$LR_{ind} = -2 \ln\left[\frac{(1-\hat{\pi}_2)^{n_{00}} \hat{\pi}_2^{n_{01}} (1-\hat{\pi}_2)^{n_{10}} \hat{\pi}_2^{n_{11}}}{(1-\hat{\pi}_{01})^{n_{00}} \hat{\pi}_{01}^{n_{01}} (1-\hat{\pi}_{11})^{n_{10}} \hat{\pi}_{11}^{n_{11}}}\right] \sim \chi^2(1)$$

O **teste conjunto** (cobertura condicional) combina:
$$LR_{cc} = LR_{uc} + LR_{ind} \sim \chi^2(2)$$

In [ ]:
def christoffersen_test(returns_arr, var_arr, alpha):
    """Teste de Christoffersen (1998) - Cobertura Condicional."""
    violations = (returns_arr < var_arr).astype(int)
    T = len(violations)

    # Contagem de transicoes
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        if violations[t-1] == 0 and violations[t] == 0: n00 += 1
        elif violations[t-1] == 0 and violations[t] == 1: n01 += 1
        elif violations[t-1] == 1 and violations[t] == 0: n10 += 1
        elif violations[t-1] == 1 and violations[t] == 1: n11 += 1

    # Probabilidades condicionais
    pi_01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    pi_11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0
    pi_2 = (n01 + n11) / (n00 + n01 + n10 + n11)  # prob incondicional

    # LR de independencia
    eps = 1e-15
    log_l0 = (n00 + n10) * np.log(1 - pi_2 + eps) + (n01 + n11) * np.log(pi_2 + eps)
    log_l1 = (n00 * np.log(1 - pi_01 + eps) + n01 * np.log(pi_01 + eps) +
              n10 * np.log(1 - pi_11 + eps) + n11 * np.log(pi_11 + eps))

    lr_ind = -2 * (log_l0 - log_l1)
    pvalue_ind = 1 - stats.chi2.cdf(lr_ind, df=1)

    # LR de cobertura incondicional (Kupiec)
    x = violations.sum()
    p_hat = x / T
    lr_uc = -2 * (x * np.log(alpha + eps) + (T - x) * np.log(1 - alpha + eps)
                  - x * np.log(p_hat + eps) - (T - x) * np.log(1 - p_hat + eps))
    pvalue_uc = 1 - stats.chi2.cdf(lr_uc, df=1)

    # LR conjunto (cobertura condicional)
    lr_cc = lr_uc + lr_ind
    pvalue_cc = 1 - stats.chi2.cdf(lr_cc, df=2)

    return {
        'lr_uc': lr_uc, 'pvalue_uc': pvalue_uc,
        'lr_ind': lr_ind, 'pvalue_ind': pvalue_ind,
        'lr_cc': lr_cc, 'pvalue_cc': pvalue_cc,
        'pi_01': pi_01, 'pi_11': pi_11,
        'n00': n00, 'n01': n01, 'n10': n10, 'n11': n11
    }

# Aplicar teste para cada modelo
print("=== Teste de Christoffersen (Cobertura Condicional) - VaR 99% ===\n")

for name, var_series in [('GARCH-Normal', var_normal),
                          ('GARCH-t', var_garch_t),
                          ('EWMA', var_ewma)]:
    result = christoffersen_test(returns.values, var_series, alpha)
    print(f"--- {name} ---")
    print(f"  Matriz de transicao: n00={result['n00']}, n01={result['n01']}, "
          f"n10={result['n10']}, n11={result['n11']}")
    print(f"  P(viol | no viol) = {result['pi_01']:.4f}")
    print(f"  P(viol | viol)    = {result['pi_11']:.4f}")
    print(f"  LR_uc  = {result['lr_uc']:.4f} (p = {result['pvalue_uc']:.4f}) "
          f"{'[REJEITA]' if result['pvalue_uc'] < 0.05 else '[OK]'}")
    print(f"  LR_ind = {result['lr_ind']:.4f} (p = {result['pvalue_ind']:.4f}) "
          f"{'[CLUSTERS]' if result['pvalue_ind'] < 0.05 else '[OK]'}")
    print(f"  LR_cc  = {result['lr_cc']:.4f} (p = {result['pvalue_cc']:.4f}) "
          f"{'[REJEITA]' if result['pvalue_cc'] < 0.05 else '[OK]'}")
    print()

## 3. Teste de Berkowitz (2001)

O teste de **Berkowitz** avalia a qualidade da distribuicao prevista inteira, nao apenas
o quantil (VaR). Usa a **Probability Integral Transform (PIT)**:

$$u_t = F_t(r_t)$$

Se o modelo for correto, $u_t \sim U(0,1)$ i.i.d. Transformando para normal:

$$z_t = \Phi^{-1}(u_t) \sim N(0,1)$$

O teste verifica se os $z_t$ seguem $N(0,1)$ via LR contra $N(\mu, \sigma^2)$ com AR(1):

$$LR = -2(L_0 - L_1) \sim \chi^2(3)$$

onde $L_0$ e a log-verossimilhanca sob $N(0,1)$ e $L_1$ sob o modelo alternativo.

In [ ]:
def berkowitz_test(returns_arr, mu_arr, sigma_arr, dist='normal', nu=None):
    """Teste de Berkowitz (2001) baseado em PIT."""
    # PIT: u_t = F_t(r_t)
    z_std = (returns_arr - mu_arr) / sigma_arr  # residuos padronizados

    if dist == 'normal':
        u_t = stats.norm.cdf(z_std)
    elif dist == 'studentt' and nu is not None:
        u_t = stats.t.cdf(z_std * np.sqrt(nu / (nu - 2)), df=nu)
    else:
        u_t = stats.norm.cdf(z_std)

    # Transformar para normal: z_t = Phi^{-1}(u_t)
    u_t = np.clip(u_t, 1e-10, 1 - 1e-10)
    z_t = stats.norm.ppf(u_t)

    # Teste: H0: z_t ~ N(0,1) iid vs H1: z_t ~ N(mu, sigma^2) com AR(1)
    T = len(z_t)

    # Log-verossimilhanca sob H0: N(0,1)
    ll_0 = np.sum(stats.norm.logpdf(z_t))

    # Log-verossimilhanca sob H1: N(mu, sigma^2) com AR(1)
    mu_hat = z_t.mean()
    z_lag = z_t[:-1]
    z_curr = z_t[1:]
    rho_hat = np.corrcoef(z_lag, z_curr)[0, 1]
    resid = z_curr - mu_hat - rho_hat * (z_lag - mu_hat)
    sigma2_hat = np.var(resid)

    ll_1 = (-0.5 * T * np.log(2 * np.pi) - 0.5 * T * np.log(sigma2_hat + 1e-15)
            - 0.5 * np.sum(resid**2) / (sigma2_hat + 1e-15))

    lr = -2 * (ll_0 - ll_1)
    pvalue = 1 - stats.chi2.cdf(lr, df=3)

    return {
        'statistic': lr, 'pvalue': pvalue, 'df': 3,
        'mu_hat': mu_hat, 'sigma_hat': np.sqrt(sigma2_hat), 'rho_hat': rho_hat
    }

# Aplicar para cada modelo
print("=== Teste de Berkowitz (PIT) ===\n")
print(f"{'Modelo':<20} {'LR':>10} {'p-valor':>10} {'mu_hat':>10} {'sigma_hat':>10} {'rho_hat':>10} {'Decisao':>12}")
print("=" * 85)

for name, mu_arr, sigma_arr, dist_name, nu_val in [
    ('GARCH-Normal', np.full(n, mu_n), res_n.conditional_volatility, 'normal', None),
    ('GARCH-t', np.full(n, mu_t), res_t.conditional_volatility, 'studentt', nu),
    ('EWMA', np.full(n, returns.mean()), res_ewma.conditional_volatility, 'normal', None)
]:
    result = berkowitz_test(returns.values, mu_arr, sigma_arr, dist_name, nu_val)
    decision = "Rejeita H0" if result['pvalue'] < 0.05 else "Nao rejeita"
    print(f"{name:<20} {result['statistic']:>10.4f} {result['pvalue']:>10.4f} "
          f"{result['mu_hat']:>10.4f} {result['sigma_hat']:>10.4f} "
          f"{result['rho_hat']:>10.4f} {decision:>12}")

print("\nSe mu_hat \u2248 0, sigma_hat \u2248 1, rho_hat \u2248 0: modelo bem especificado")
print("mu_hat != 0: vies na media. sigma_hat != 1: erro na volatilidade. rho_hat != 0: dependencia.")

## 4. Traffic Light System (Basel)

O **sistema de semaforo** do Comite de Basileia classifica modelos de VaR com base
no numero de violacoes em 250 dias de negociacao (VaR 99%):

| Zona | Violacoes (em 250 dias) | Acao |
|------|------------------------|------|
| **Verde** | 0 - 4 | Modelo aceito, sem penalidade |
| **Amarelo** | 5 - 9 | Investigacao necessaria, multiplicador aumenta |
| **Vermelho** | 10+ | Modelo rejeitado, penalidade severa |

O multiplicador de capital aumenta com o numero de violacoes:
- 4 ou menos: multiplicador = 3.0x
- 5: 3.4x, 6: 3.5x, 7: 3.65x, 8: 3.75x, 9: 3.85x
- 10+: 4.0x

In [ ]:
def basel_traffic_light(returns_arr, var_arr, window=250):
    """Sistema de semaforo do Basel para VaR 99%."""

    T = len(returns_arr)
    colors = []
    n_violations_list = []

    for t in range(window, T):
        window_returns = returns_arr[t - window:t]
        window_var = var_arr[t - window:t]
        n_viol = (window_returns < window_var).sum()
        n_violations_list.append(n_viol)

        if n_viol <= 4:
            colors.append('green')
        elif n_viol <= 9:
            colors.append('yellow')
        else:
            colors.append('red')

    return colors, n_violations_list

# Aplicar para cada modelo
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, (name, var_series) in zip(axes, [('GARCH-Normal', var_normal),
                                           ('GARCH-t', var_garch_t),
                                           ('EWMA', var_ewma)], strict=False):
    colors, n_viols = basel_traffic_light(returns.values, var_series, window=250)
    idx = returns.index[250:]
    n_viols = np.array(n_viols)

    # Colorir fundo por zona
    ax.fill_between(idx, 0, 4, alpha=0.1, color='green', label='Verde (0-4)')
    ax.fill_between(idx, 4, 9, alpha=0.1, color='yellow', label='Amarelo (5-9)')
    ax.fill_between(idx, 9, max(n_viols.max(), 12), alpha=0.1, color='red', label='Vermelho (10+)')

    ax.plot(idx, n_viols, color='black', linewidth=1)
    ax.axhline(y=2.5, color='green', linestyle=':', alpha=0.5, label=f'Esperado: {250*0.01:.1f}')
    ax.set_title(f'Traffic Light - {name}')
    ax.set_ylabel('Violacoes (250 dias)')
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Data')
plt.tight_layout()
plt.show()

# Resumo final
print("=== Traffic Light - Ultimo periodo de 250 dias ===\n")
for name, var_series in [('GARCH-Normal', var_normal),
                          ('GARCH-t', var_garch_t),
                          ('EWMA', var_ewma)]:
    last_250_ret = returns.values[-250:]
    last_250_var = var_series[-250:]
    n_viol = (last_250_ret < last_250_var).sum()
    if n_viol <= 4: zone = "VERDE"
    elif n_viol <= 9: zone = "AMARELO"
    else: zone = "VERMELHO"
    print(f"  {name:<20}: {n_viol} violacoes -> Zona {zone}")

## 5. Backtesting comparativo

Agora vamos comparar **todos os metodos de VaR** usando os testes de backtesting.
Incluimos VaR Normal (estatico), GARCH-Normal, GARCH-t, EWMA e FHS.

In [ ]:
# Gerar todas as series de VaR (95% e 99%)
mu_const = returns.mean()
sigma_const = returns.std()

# VaR Normal (estatico)
var_static_95 = np.full(n, mu_const + stats.norm.ppf(0.05) * sigma_const)
var_static_99 = np.full(n, mu_const + stats.norm.ppf(0.01) * sigma_const)

# VaR GARCH-Normal
var_gn_95 = mu_n + stats.norm.ppf(0.05) * res_n.conditional_volatility
var_gn_99 = var_normal  # ja calculado

# VaR GARCH-t
t_95 = stats.t.ppf(0.05, df=nu) * np.sqrt((nu - 2) / nu)
t_99 = stats.t.ppf(0.01, df=nu) * np.sqrt((nu - 2) / nu)
var_gt_95 = mu_t + t_95 * res_t.conditional_volatility
var_gt_99 = var_garch_t  # ja calculado

# VaR EWMA
var_ew_95 = mu_const + stats.norm.ppf(0.05) * res_ewma.conditional_volatility
var_ew_99 = var_ewma  # ja calculado

# VaR FHS
resid_std = res_n.resid
q_z_05 = np.percentile(resid_std, 5)
q_z_01 = np.percentile(resid_std, 1)
var_fhs_95 = mu_n + res_n.conditional_volatility * q_z_05
var_fhs_99 = mu_n + res_n.conditional_volatility * q_z_01

# Backtesting comparativo - VaR 99%
models_99 = {
    'Normal (estatico)': var_static_99,
    'GARCH-Normal': var_gn_99,
    'GARCH-t': var_gt_99,
    'EWMA (\u03bb=0.94)': var_ew_99,
    'FHS': var_fhs_99
}

print("=" * 100)
print(f"{'BACKTESTING COMPARATIVO - VaR 99%':^100}")
print("=" * 100)
print(f"{'Modelo':<20} {'Viol':>6} {'Taxa':>8} {'Kupiec':>10} {'p-val':>8} "
      f"{'Christ.':>10} {'p-val':>8} {'Semaforo':>10}")
print("-" * 100)

for name, var_series in models_99.items():
    kup = kupiec_test(returns.values, var_series, 0.01)
    chris = christoffersen_test(returns.values, var_series, 0.01)
    n_viol_last = (returns.values[-250:] < var_series[-250:]).sum()
    if n_viol_last <= 4: zone = "Verde"
    elif n_viol_last <= 9: zone = "Amarelo"
    else: zone = "Vermelho"

    print(f"{name:<20} {kup['violations']:>6} {kup['rate']:>8.4f} "
          f"{kup['statistic']:>10.4f} {kup['pvalue']:>8.4f} "
          f"{chris['lr_cc']:>10.4f} {chris['pvalue_cc']:>8.4f} {zone:>10}")

print("=" * 100)
print(f"{'Esperado':<20} {n*0.01:>6.0f} {'0.0100':>8}")

# Grafico comparativo
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(returns.index, returns.values, color='gray', alpha=0.3, linewidth=0.5, label='Retornos')

colors = ['blue', 'green', 'red', 'orange', 'purple']
for (name, var_series), color in zip(models_99.items(), colors, strict=False):
    ax.plot(returns.index, var_series, color=color, linewidth=0.8, alpha=0.8, label=name)

mask = returns.values < var_gt_99
ax.scatter(returns.index[mask], returns.values[mask],
           color='red', s=10, zorder=5, alpha=0.6, label='Violacoes (GARCH-t)')

ax.set_title('Backtesting Comparativo - VaR 99%')
ax.set_ylabel('Retorno / VaR')
ax.legend(loc='lower left', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 6. Dynamic Quantile (DQ) test

O teste **DQ** de Engle e Manganelli (2004) verifica se a sequencia de violacoes e
previsivel por variaveis defasadas. A regressao:

$$I_t - \alpha = \beta_0 + \beta_1 I_{t-1} + \cdots + \beta_K I_{t-K} + \beta_{K+1} \text{VaR}_t + u_t$$

onde $I_t = \mathbb{1}(r_t < \text{VaR}_t) - \alpha$ e o indicador de violacao centralizado.

**Hipotese nula**: $\beta_0 = \beta_1 = \cdots = \beta_{K+1} = 0$ (violacoes nao previsiveis).

A estatistica e:

$$DQ = \frac{\hat{\beta}' X'X \hat{\beta}}{\alpha(1-\alpha)} \sim \chi^2(K+2)$$

Se o modelo for bom, as violacoes passadas e o nivel do VaR nao devem prever violacoes futuras.

In [ ]:
def dq_test(returns_arr, var_arr, alpha, K=4):
    """Dynamic Quantile test de Engle e Manganelli (2004)."""
    T = len(returns_arr)
    hit = (returns_arr < var_arr).astype(float) - alpha  # hit centralizado

    # Construir matriz de regressores: constante + K lags de hit + VaR
    X = np.ones((T - K, 1))  # constante
    for k in range(1, K + 1):
        X = np.hstack([X, hit[K - k:T - k].reshape(-1, 1)])
    X = np.hstack([X, var_arr[K:].reshape(-1, 1)])  # VaR como regressor

    y = hit[K:]

    # Regressao OLS
    beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]

    # Estatistica DQ
    dq_stat = (beta_hat @ X.T @ X @ beta_hat) / (alpha * (1 - alpha))
    df = K + 2  # constante + K lags + VaR
    pvalue = 1 - stats.chi2.cdf(dq_stat, df=df)

    return {'statistic': dq_stat, 'pvalue': pvalue, 'df': df, 'beta': beta_hat}

# Aplicar DQ test para cada modelo
print("=== Dynamic Quantile (DQ) Test - Engle & Manganelli (2004) ===\n")
print(f"{'Modelo':<20} {'DQ stat':>10} {'df':>4} {'p-valor':>10} {'Decisao':>15}")
print("=" * 62)

for name, var_series in models_99.items():
    result = dq_test(returns.values, var_series, alpha=0.01, K=4)
    decision = "Rejeita H0" if result['pvalue'] < 0.05 else "Nao rejeita"
    print(f"{name:<20} {result['statistic']:>10.4f} {result['df']:>4} "
          f"{result['pvalue']:>10.4f} {decision:>15}")

print("\nH0: violacoes nao sao previsiveis pelas violacoes passadas nem pelo VaR")
print("Rejeicao indica que o modelo nao captura toda a dinamica de risco")

# Resumo final visual
print("\n" + "=" * 80)
print(f"{'RANKING FINAL DOS MODELOS':^80}")
print("=" * 80)
print("\nO melhor modelo de VaR deve:")
print("  1. Nao rejeitar Kupiec (taxa de violacao correta)")
print("  2. Nao rejeitar Christoffersen (violacoes independentes)")
print("  3. Nao rejeitar DQ (violacoes nao previsiveis)")
print("  4. Estar na zona Verde do semaforo Basel")
print("\nEm geral, GARCH-t e FHS tendem a ter melhor performance,")
print("especialmente ao nivel de 99%, onde caudas pesadas importam.")